# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json.get('name')}: {metadata_json.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

> **Note:** All references to record sets, fields, and columns are by their `@id`.

In [ ]:
# List all record set `@id`s and their fields
print("Available Record Sets (by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name','N/A')}")
    print("  Fields:")
    for fld in rs.get('field', []):
        # Field could be either a dict or an @id ref
        if isinstance(fld, dict):
            print(f"      - {fld.get('@id')} (name: {fld.get('name')})")
        else:
            print(f"      - {fld}")
    print()

# For demonstration: print the first 1-2 records of each record set (using @id)
for rs in record_sets:
    print(f"--- Sample records from {rs['@id']} ---")
    try:
        for i, row in enumerate(dataset.records(record_set=rs['@id'])):
            print(row)
            if i >= 1:
                break
    except Exception as e:
        print(f"   Error loading records for {rs['@id']}: {e}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> If you want to extract all tables and inspect them, you can loop using the list of record set `@id`s.

In [ ]:
# Extract data from each record set into a DataFrame
from collections import OrderedDict

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = OrderedDict()

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"[Error] Could not load DataFrame for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note:** In your own project, choose fields that make sense for your analysis. Here we show an example EDA process with the first available numeric field (if any) for the main data table.

In [ ]:
# Pick a primary record set (e.g., the main protein table) for in-depth EDA
import numpy as np

if len(dataframes) == 0:
    print("No dataframes found!")
else:
    # Typically the main data is in the largest table
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    main_df = dataframes[main_rs_id]
    print(f"Using record set: {main_rs_id} for EDA")
    print(main_df.info())

    # Find first numeric column
    numeric_field_id = None
    for col in main_df.columns:
        sample_vals = main_df[col].dropna()[:10]
        # Try casting
        try:
            nums = pd.to_numeric(sample_vals)
            if nums.notna().all():
                numeric_field_id = col
                break
        except Exception:
            continue
    if numeric_field_id is None:
        print("Did not find a numeric field to analyze.")
    else:
        # Convert column if necessary
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

        threshold = np.nanpercentile(main_df[numeric_field_id], 75)  # Use Q3 as example threshold
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} records\n")

        # Normalize
        col_mean = filtered_df[numeric_field_id].mean()
        col_std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - col_mean) / col_std

        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping by a categorical field, pick the first non-numeric
        group_field = None
        for col in main_df.columns:
            if col != numeric_field_id and main_df[col].dtype == object:
                group_field = col
                # Should have non-unique values
                if main_df[col].nunique() < len(main_df) * 0.9:
                    break
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by '{group_field}':")
            print(grouped.head())
        else:
            print("No suitable categorical group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Example: Histogram of the selected numeric field. Modify for your analysis as needed.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) and numeric_field_id:
    plt.figure(figsize=(8,4))
    main_df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id} in record set {main_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        # Boxplot by group
        plt.figure(figsize=(10,5))
        main_df.boxplot(column=numeric_field_id, by=group_field, rot=90)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle('')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Used `mlcroissant` to access the dataset via its Croissant schema URL.
- Inspected available record sets and schema, loaded data into pandas DataFrames.
- Performed exploratory steps on a selected primary table, filtering and normalizing a numeric field.
- Visualized value distributions and grouped summaries.

You can now continue with additional analyses, machine learning, or export these DataFrames as needed!